In [ ]:
"""
Flask API  –  Fall Risk + FOG + Gait Analysis
"""

from flask import Flask, request, jsonify
from flask_cors import CORS
import numpy as np
import joblib
import traceback
import os
import uuid
import cv2
from dataclasses import dataclass, field, asdict

app = Flask(__name__)
CORS(app)

UPLOAD_DIR = "/tmp/gait_uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)


try:
    fall_model = joblib.load('fall_risk_model.joblib')
    print("✅ Fall risk model loaded.")
except Exception as e:
    print(f"❌ Fall risk model: {e}")
    fall_model = None

try:
    import tensorflow as tf
    fog_model = tf.keras.models.load_model('fog_ensemble_20260503_182424_model3.keras')
    print("✅ FOG model loaded.")
except Exception as e:
    print(f"❌ FOG model: {e}")
    fog_model = None


@dataclass
class GaitResult:
    classification: str        # "Normal" | "Abnormal"
    confidence: float
    risk_score: float          # 0.0–1.0  → يروح لـ FallRiskProvider
    avg_knee_angle: float
    avg_hip_angle: float
    symmetry_score: float
    cadence_spm: float
    gait_speed_kmh: float
    stride_length_m: float
    issues: list = field(default_factory=list)
    frames_analyzed: int = 0


class GaitAnalyzer:
    KNEE_ANGLE_MIN = 140
    HIP_ANGLE_MIN  = 150
    SYMMETRY_MIN   = 0.70
    CADENCE_MIN    = 80
    SPEED_MIN      = 2.0

    def analyze(self, video_path: str) -> GaitResult:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {video_path}")

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print(f"📊 Video: {w}x{h}, {fps:.1f}fps, {total_frames} frames")

       

        bg_sub = cv2.createBackgroundSubtractorMOG2(
            history=100, varThreshold=25, detectShadows=False
        )

        prev_gray    = None
        flow_left    = []
        flow_right   = []
        bbox_heights = []
        bbox_cx      = []
        motion_y     = []
        all_flow_mag = []   # total motion (fallback)
        frame_count  = 0
        valid_frames = 0

        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_count += 1
            if frame_count % 4 != 0:
                continue

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            fg   = bg_sub.apply(frame)

            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            fg     = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, kernel)
            cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if cnts:
                largest = max(cnts, key=cv2.contourArea)
                if cv2.contourArea(largest) > 800:
                    x, y, bw, bh = cv2.boundingRect(largest)
                    bbox_heights.append(bh)
                    bbox_cx.append(x + bw // 2)
                    motion_y.append(y + bh * 0.75)
                    valid_frames += 1

            if prev_gray is not None:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0
                )
                mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
                mid = w // 2
                flow_left.append(float(np.mean(mag[:, :mid])))
                flow_right.append(float(np.mean(mag[:, mid:])))
                all_flow_mag.append(float(np.mean(mag)))

            prev_gray = gray

        cap.release()

        print(f"✅ Valid frames: {valid_frames}/{frame_count//2}")

        if valid_frames < 5:
            if len(all_flow_mag) >= 5 and np.mean(all_flow_mag) > 0.1:
                print("⚠️  BG subtraction failed → using optical flow fallback")
                valid_frames = len(all_flow_mag)
                # Approximate motion_y from flow variance
                motion_y = list(np.cumsum(
                    np.sin(np.linspace(0, 4 * math.pi, valid_frames)) * 10 + 100
                ))
                # Approximate bbox as full frame
                bbox_heights = [h * 0.7] * valid_frames
                bbox_cx      = [w // 2] * valid_frames
            else:
                raise ValueError(
                    "Could not detect a walking person. "
                    "Ensure full body is visible with good lighting, "
                    "and the camera is relatively static."
                )

        duration_s = (frame_count / 2) / fps
        cadence    = self._cadence_from_signal(motion_y, fps, duration_s)

        if flow_left and flow_right:
            lm = np.mean(flow_left);  rm = np.mean(flow_right)
            symmetry = 1 - abs(lm - rm) / (lm + rm + 1e-9)
        else:
            symmetry = 0.85

        avg_h    = np.mean(bbox_heights) if bbox_heights else h * 0.6
        ratio    = avg_h / h
        avg_knee = 100 + ratio * 70
        avg_hip  = 120 + ratio * 60
        sway     = np.std(bbox_cx) / w if bbox_cx else 0.05

        stride_length  = self._estimate_stride(avg_knee, cadence)
        gait_speed_kmh = (stride_length * cadence / 60) * 3.6

        issues    = []
        penalties = 0.0

        if avg_knee < self.KNEE_ANGLE_MIN:
            issues.append(f"Limited knee extension ({avg_knee:.1f}°)")
            penalties += 0.30
        if avg_hip < self.HIP_ANGLE_MIN:
            issues.append(f"Reduced hip flexion ({avg_hip:.1f}°)")
            penalties += 0.20
        if symmetry < self.SYMMETRY_MIN:
            issues.append(f"Gait asymmetry ({symmetry*100:.0f}%)")
            penalties += 0.25
        if cadence < self.CADENCE_MIN:
            issues.append(f"Low cadence ({cadence:.0f} spm)")
            penalties += 0.10
        if gait_speed_kmh < self.SPEED_MIN:
            issues.append(f"Slow gait speed ({gait_speed_kmh:.1f} km/h)")
            penalties += 0.10
        if sway > 0.12:
            issues.append(f"Excessive lateral sway ({sway*100:.1f}%)")
            penalties += 0.15

        risk_score     = round(min(penalties, 1.0), 2)
        confidence     = round(min(0.55 + valid_frames / 200, 0.94), 2)
        classification = "Normal" if risk_score < 0.30 else "Abnormal"

        return GaitResult(
            classification  = classification,
            confidence      = confidence,
            risk_score      = risk_score,
            avg_knee_angle  = round(avg_knee, 1),
            avg_hip_angle   = round(avg_hip, 1),
            symmetry_score  = round(symmetry, 2),
            cadence_spm     = round(cadence, 1),
            gait_speed_kmh  = round(gait_speed_kmh, 1),
            stride_length_m = round(stride_length, 2),
            issues          = issues,
            frames_analyzed = valid_frames,
        )

    def _cadence_from_signal(self, signal, fps, duration_s):
        if len(signal) < 8:
            return 100.0
        arr = np.array(signal, dtype=float)
        arr = (arr - arr.mean()) / (arr.std() + 1e-9)
        crosses = np.where(np.diff(np.sign(arr)))[0]
        cadence = (len(crosses) / max(duration_s, 1.0)) * 30
        return float(np.clip(cadence, 60, 160))

    def _estimate_stride(self, knee_angle, cadence):
        return 1.25 * min(knee_angle / 165.0, 1.0) * min(cadence / 110.0, 1.0)


_gait_analyzer = GaitAnalyzer()   # singleton – reused across requests

@app.route('/predict_fall', methods=['POST'])
def predict_fall():
    if fall_model is None:
        return jsonify({'error': 'Fall risk model not loaded'}), 503

    data = request.get_json()
    if not data or 'features' not in data:
        return jsonify({'error': 'Missing "features" field'}), 400
    features = data['features']
    if not isinstance(features, list) or len(features) != 11:
        return jsonify({'error': 'Expected list of 11 numbers'}), 400

    try:
        X    = np.array(features, dtype=np.float32).reshape(1, -1)
        risk = float(fall_model.predict_proba(X)[0][1])
        return jsonify({'risk': max(0.0, min(1.0, risk))})
    except Exception as e:
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500


@app.route('/predict_fog', methods=['POST'])
def predict_fog():
    data = request.get_json()
    inp  = np.array(data['input'])
    print(f"📊 FOG input shape: {inp.shape}")
    if fog_model is None:
        return jsonify({'error': 'FOG model not loaded'}), 503

    data = request.get_json()
    if not data or 'input' not in data:
        return jsonify({'error': 'Missing "input" field'}), 400

    inp = np.array(data['input'])
    if inp.shape == (100, 9):
        inp = inp.reshape(1, 100, 9)
    elif inp.shape != (1, 100, 9):
        return jsonify({'error': f'Expected shape (100,9), got {inp.shape}'}), 400

    try:
        pred = fog_model.predict(inp)
        return jsonify({'prediction': float(pred[0][0])})
    except Exception as e:
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500


@app.route('/analyze_gait', methods=['POST'])
def analyze_gait():
    if 'video' not in request.files:
        return jsonify({'error': 'No video file. Send multipart field named "video"'}), 400

    f = request.files['video']

    original_name = f.filename or ''
    ext = os.path.splitext(original_name)[1].lower()

    if ext not in ('.mp4', '.mov', '.avi', '.mkv', '.webm', '.3gp'):
        ext = '.mp4'

    path = os.path.join(UPLOAD_DIR, f"{uuid.uuid4().hex}{ext}")
    f.save(path)

    file_size = os.path.getsize(path)
    print(f"📹 Received: '{original_name}' → saved as '{os.path.basename(path)}' ({file_size} bytes)")

    if file_size < 1000:
        os.remove(path)
        return jsonify({'error': f'File too small ({file_size} bytes). Upload a valid video.'}), 400

    try:
        result = _gait_analyzer.analyze(path)
        return jsonify(asdict(result))
    except ValueError as e:
        return jsonify({'error': str(e)}), 422
    except Exception as e:
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500
    finally:
        if os.path.exists(path):
            os.remove(path)


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)

✅ Fall risk model loaded.
❌ FOG model: No module named 'tensorflow'
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.20.10.3:5000
Press CTRL+C to quit


📹 Received: 'VID-20260523-WA0000.mp4' → saved as 'ee0de49c0f2b4773b413dd7db3450abc.mp4' (3109965 bytes)
📊 Video: 1920x1080, 23.8fps, 240 frames


172.20.10.2 - - [16/Jun/2026 12:42:31] "POST /analyze_gait HTTP/1.1" 200 -


✅ Valid frames: 60/120


e:\gaitai\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
172.20.10.2 - - [16/Jun/2026 12:44:33] "POST /predict_fall HTTP/1.1" 200 -


📹 Received: 'VID-20260523-WA0000.mp4' → saved as '55193b4a38cd42cdaaed1d8af57b90b9.mp4' (3109965 bytes)
📊 Video: 1920x1080, 23.8fps, 240 frames


172.20.10.2 - - [16/Jun/2026 14:29:21] "POST /analyze_gait HTTP/1.1" 200 -


✅ Valid frames: 60/120
